# Segundo parcial - Cloud Provider Analytics
**Jose Emiliano Villaverde**  
Minería de Datos II - ISTEA

Leo los CSV y los eventos, limpio los datos y obtengo costos diarios.
Al final cargo los resultados y hago dos consultas en mi propia base Astra.

En Colab elegir **Entorno de ejecución > Cambiar tipo de entorno > versión 2026.07**
para trabajar con Python 3.12. Usar CPU, sin GPU.

Ejecución completa en Colab del 18/11/2026. Incluye las tres variables de Silver,
la bandera de costo inválido y los resultados reales de mi base Astra.

## 1. Instalar las librerías

Uso Spark para procesar y el driver Cassandra para la conexión del final.

In [1]:
import sys
print("Python:", sys.version.split()[0])
assert sys.version_info[:2] == (3, 12), "En Colab elegir versión 2026.07"
%pip install -q pyspark==3.5.3 cassandra-driver==3.29.2

Python: 3.12.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


## 2. Iniciar Spark

Elijo UTC para que las fechas de los eventos se interpreten siempre igual.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, json, zipfile

spark = (SparkSession.builder.appName("MD2 - Jose Villaverde").master("local[2]")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.ansi.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

Spark: 3.5.3


## 3. Subir el dataset

Elijo el ZIP original que entregó el profesor.

In [3]:
from google.colab import files
subidos = files.upload()
archivo_zip = next(iter(subidos))

Saving cloud_provider_challenge_dataset_v1.zip to cloud_provider_challenge_dataset_v1.zip


## 4. Preparar las carpetas

Los originales van a Landing. No borro las carpetas cuando repito una ejecución.

In [4]:
BASE = "/content/md2_jose"
LANDING = BASE + "/datalake/landing"
BRONZE = BASE + "/datalake/bronze"
SILVER = BASE + "/datalake/silver"
GOLD = BASE + "/datalake/gold"
QUARANTINE = BASE + "/datalake/quarantine"
CHECKPOINT = BASE + "/checkpoints"
EVIDENCIA = BASE + "/evidencia"
for carpeta in [BRONZE, SILVER, GOLD, QUARANTINE, CHECKPOINT, EVIDENCIA]:
    os.makedirs(carpeta, exist_ok=True)
if not os.path.exists(LANDING):
    with zipfile.ZipFile(archivo_zip) as z:
        z.extractall(BASE)
print("Landing:", os.listdir(LANDING))

Landing: ['marketing_touches.csv', 'billing_monthly.csv', 'customers_orgs.csv', 'resources.csv', 'support_tickets.csv', 'users.csv', 'usage_events_stream', 'nps_surveys.csv']


## 5. Agregar datos de origen

Esta es la única función auxiliar del procesamiento. La uso para agregar origen y hora a los tres CSV.

In [5]:
def agregar_origen(df, archivo):
    return (df.withColumn("ingest_ts", F.current_timestamp())
            .withColumn("source_file", F.lit(archivo))
            .withColumn("ingest_date", F.current_date()))

## 6. Leer y guardar clientes

El esquema respeta el orden real del CSV. Limpio duplicados por org_id y escribo Bronze.

In [6]:
esquema_clientes = """org_id STRING, org_name STRING, industry STRING,
hq_region STRING, plan_tier STRING, is_enterprise BOOLEAN, signup_date DATE,
sales_rep STRING, lifecycle_stage STRING, marketing_source STRING, nps_score DOUBLE"""
clientes = (spark.read.option("header", True).option("enforceSchema", False)
            .schema(esquema_clientes).csv(LANDING + "/customers_orgs.csv")
            .dropDuplicates(["org_id"]))
clientes = agregar_origen(clientes, "customers_orgs.csv")
clientes.write.mode("overwrite").partitionBy("ingest_date").parquet(BRONZE + "/customers_orgs")
print("Clientes:", clientes.count())

Clientes: 80


## 7. Leer y guardar usuarios

Los booleanos y fechas tienen tipos definidos; no uso inferSchema.

In [7]:
esquema_usuarios = """user_id STRING, org_id STRING, email STRING, role STRING,
active BOOLEAN, created_at TIMESTAMP, last_login TIMESTAMP"""
usuarios = (spark.read.option("header", True).option("enforceSchema", False)
            .schema(esquema_usuarios).csv(LANDING + "/users.csv")
            .dropDuplicates(["user_id"]))
usuarios = agregar_origen(usuarios, "users.csv")
usuarios.write.mode("overwrite").partitionBy("ingest_date").parquet(BRONZE + "/users")
print("Usuarios:", usuarios.count())

Usuarios: 800


## 8. Leer y guardar facturas

La clave es invoice_id. El archivo completo tiene 240 facturas, no tres.

In [8]:
esquema_facturas = """invoice_id STRING, org_id STRING, month DATE,
subtotal DOUBLE, credits DOUBLE, taxes DOUBLE, currency STRING, exchange_rate_to_usd DOUBLE"""
facturas = (spark.read.option("header", True).option("enforceSchema", False)
            .schema(esquema_facturas).csv(LANDING + "/billing_monthly.csv")
            .dropDuplicates(["invoice_id"]))
facturas = agregar_origen(facturas, "billing_monthly.csv")
facturas.write.mode("overwrite").partitionBy("month").parquet(BRONZE + "/billing_monthly")
print("Facturas:", facturas.count())

Facturas: 240


## 9. Leer eventos con Structured Streaming

El campo value puede venir como texto. Los campos de v2 quedan nulos cuando no existen en v1.

In [9]:
esquema_eventos = """event_id STRING, timestamp STRING, org_id STRING,
resource_id STRING, service STRING, region STRING, metric STRING, value STRING,
unit STRING, cost_usd_increment DOUBLE, schema_version INT, genai_tokens BIGINT, carbon_kg DOUBLE"""
flujo = (spark.readStream.schema(esquema_eventos).option("maxFilesPerTrigger", 20)
         .json(LANDING + "/usage_events_stream")
         .withColumn("event_ts", F.to_timestamp("timestamp"))
         .withColumn("event_date", F.to_date("event_ts"))
         .withColumn("ingest_ts", F.current_timestamp())
         .withColumn("source_file", F.input_file_name())
         .withWatermark("event_ts", "90 days")
         .dropDuplicatesWithinWatermark(["event_id"]))

## 10. Guardar Bronze de eventos

availableNow procesa lo pendiente y termina. El checkpoint recuerda los archivos procesados. Uso 90 días porque el histórico viene mezclado a lo largo de unos dos meses. La prueba pequeña de más adelante usa 10 minutos.

In [10]:
consulta = (flujo.writeStream.format("parquet").partitionBy("event_date")
            .option("path", BRONZE + "/usage_events")
            .option("checkpointLocation", CHECKPOINT + "/eventos")
            .outputMode("append").trigger(availableNow=True).start())
consulta.awaitTermination()
eventos = spark.read.parquet(BRONZE + "/usage_events")
print("Eventos Bronze:", eventos.count())
eventos.select("event_id", "event_ts", "service", "value").show(5)

Eventos Bronze: 43200
+----------------+-------------------+----------+-------+
|        event_id|           event_ts|   service|  value|
+----------------+-------------------+----------+-------+
|evt_g77ocsgw7441|2025-07-28 05:54:00|   compute|  105.0|
|evt_x8ggun4538bo|2025-07-28 02:54:00|   storage|   87.0|
|evt_cbzjeci3pizk|2025-07-28 01:37:00|     genai|14.7184|
|evt_51xv5ku38yiv|2025-07-28 06:59:00|networking|  134.0|
|evt_r1psluxp2vlj|2025-07-28 10:12:00| analytics|  121.0|
+----------------+-------------------+----------+-------+
only showing top 5 rows



## 11. Revisar la calidad

Compruebo ID, costo, unidad y conversión de value. `costo_invalido` es una
bandera booleana: marca costos nulos, NaN o menores a -0.01.
Escribo el primer motivo encontrado y conservo las filas descartadas en Quarantine.
La unicidad de event_id se controla con la deduplicación y la prueba del paso 17.

In [11]:
eventos = eventos.withColumn("valor_numero", F.expr("try_cast(value as double)"))
eventos = eventos.withColumn("costo_invalido",
    F.col("cost_usd_increment").isNull() | F.isnan("cost_usd_increment") |
    (F.col("cost_usd_increment") < -0.01))
revisados = eventos.withColumn("motivo",
    F.when(F.col("event_id").isNull(), "ID nulo")
     .when(F.col("event_ts").isNull(), "fecha inválida")
     .when(F.col("costo_invalido"), "costo inválido")
     .when(F.col("value").isNotNull() & F.col("unit").isNull(), "value sin unit")
     .when(F.col("value").isNotNull() & F.col("valor_numero").isNull(), "value no numérico"))
validos = revisados.filter(F.col("motivo").isNull())
quarantine = revisados.filter(F.col("motivo").isNotNull())
quarantine.write.mode("overwrite").parquet(QUARANTINE)
print("Válidos:", validos.count(), "Quarantine:", quarantine.count())
quarantine.select("event_id", "motivo", "costo_invalido").show(5)
print("Filas con costo inválido:", revisados.filter("costo_invalido").count())
quarantine.filter("costo_invalido").select("event_id", "cost_usd_increment", "costo_invalido").show(3)

Válidos: 40956 Quarantine: 2244
+----------------+--------------+--------------+
|        event_id|        motivo|costo_invalido|
+----------------+--------------+--------------+
|evt_x8ggun4538bo|value sin unit|         false|
|evt_40qkafeqknqq|value sin unit|         false|
|evt_gset19zyfqbg|value sin unit|         false|
|evt_w7hlavlc8t6o|value sin unit|         false|
|evt_0vjthzcb8f92|value sin unit|         false|
+----------------+--------------+--------------+
only showing top 5 rows

Filas con costo inválido: 211
+----------------+------------------+--------------+
|        event_id|cost_usd_increment|costo_invalido|
+----------------+------------------+--------------+
|evt_s1zbencnsynd|           -0.1399|          true|
|evt_3rygp8arcgcj|           -2.7801|          true|
|evt_fac9pru3xlpd|           -0.5116|          true|
+----------------+------------------+--------------+
only showing top 3 rows



## 12. Limpiar clientes y unir los datos

Normalizo región y plan. Uso left join para conservar los eventos aunque no encuentre al cliente.

In [12]:
clientes_limpios = (spark.read.parquet(BRONZE + "/customers_orgs")
                   .withColumn("hq_region", F.lower(F.trim("hq_region")))
                   .withColumn("plan_tier", F.lower(F.trim("plan_tier")))
                   .withColumn("org_name", F.trim("org_name")))
clientes_limpios.write.mode("overwrite").parquet(SILVER + "/customers_orgs")
dimension = clientes_limpios.select("org_id", "org_name", "hq_region", "plan_tier")
silver = validos.join(dimension, "org_id", "left")

## 13. Crear variables y guardar Silver

Creo tres variables numéricas a partir de metric y value: requests, cpu_hours
y storage_gb_hours. Cada evento aporta a la variable que corresponde a su métrica;
las otras quedan en cero. Conservo tokens y carbono desconocidos como nulos.
El costo diario se calcula en Gold.

In [13]:
silver = (silver.withColumn("service", F.lower(F.trim("service")))
          .withColumn("requests", F.when(F.col("metric") == "requests", F.col("valor_numero")).otherwise(0))
          .withColumn("cpu_hours", F.when(F.col("metric") == "cpu_hours", F.col("valor_numero")).otherwise(0))
          .withColumn("storage_gb_hours", F.when(F.col("metric") == "storage_gb_hours", F.col("valor_numero")).otherwise(0)))
silver.write.mode("overwrite").partitionBy("event_date").parquet(SILVER + "/usage_events")
print("Silver:", silver.count())
silver.select("org_id", "service", "requests", "cpu_hours", "storage_gb_hours").show(5)

Silver: 40956
+------------+----------+--------+---------+----------------+
|      org_id|   service|requests|cpu_hours|storage_gb_hours|
+------------+----------+--------+---------+----------------+
|org_1n13jcat|   compute|   105.0|      0.0|             0.0|
|org_pbhsahxt|     genai|     0.0|      0.0|         14.7184|
|org_kdgigatj|networking|   134.0|      0.0|             0.0|
|org_om5igqpk| analytics|   121.0|      0.0|             0.0|
|org_4zw9xa3k| analytics|     0.0|   2.2489|             0.0|
+------------+----------+--------+---------+----------------+
only showing top 5 rows



## 14. Calcular Gold

Agrupo por organización, día y servicio. Sumo costos, requests, tokens y carbono. event_count indica cuántos eventos entraron en cada grupo.

In [14]:
silver_guardado = spark.read.parquet(SILVER + "/usage_events")
gold = (silver_guardado.groupBy("org_id", "event_date", "service")
        .agg(F.round(F.sum("cost_usd_increment"), 6).alias("daily_cost_usd"),
             F.sum("requests").alias("requests"),
             F.sum("genai_tokens").alias("genai_tokens"),
             F.sum("carbon_kg").alias("carbon_kg"), F.count("*").alias("event_count")))
gold.write.mode("overwrite").partitionBy("event_date").parquet(GOLD + "/org_daily_usage_by_service")
print("Gold:", gold.count())
gold.orderBy("org_id", "event_date", "service").show(5)

Gold: 11050
+------------+----------+---------+--------------+--------+------------+---------+-----------+
|      org_id|event_date|  service|daily_cost_usd|requests|genai_tokens|carbon_kg|event_count|
+------------+----------+---------+--------------+--------+------------+---------+-----------+
|org_0lvsnujz|2025-07-03|analytics|        8.8739|   116.0|        NULL|     NULL|          3|
|org_0lvsnujz|2025-07-03| database|        0.5723|     0.0|        NULL|     NULL|          2|
|org_0lvsnujz|2025-07-03|  storage|        5.5542|   255.0|        NULL|     NULL|          3|
|org_0lvsnujz|2025-07-04|analytics|        3.2915|     0.0|        NULL|     NULL|          3|
|org_0lvsnujz|2025-07-04| database|       12.7337|   235.0|        NULL|     NULL|          6|
+------------+----------+---------+--------------+--------+------------+---------+-----------+
only showing top 5 rows



## 15. Mirar los costos altos

Como análisis inicial, marco los costos diarios mayores al percentil 99. No son necesariamente errores; el criterio puede favorecer a servicios más caros.


In [15]:
umbral = gold.agg(F.percentile_approx("daily_cost_usd", 0.99)).first()[0]
costos_altos = gold.withColumn("costo_alto", F.col("daily_cost_usd") > umbral)
costos_altos.write.mode("overwrite").parquet(GOLD + "/costos_altos")
print("Umbral:", umbral, "Filas marcadas:", costos_altos.filter("costo_alto").count())

Umbral: 61.2773 Filas marcadas: 111


## 16. Comprobar los conteos

Estas cuentas ayudan a detectar pérdida o multiplicación de registros.

In [16]:
assert eventos.count() == validos.count() + quarantine.count()
assert gold.agg(F.sum("event_count")).first()[0] == silver_guardado.count()
assert gold.count() == gold.select("org_id", "event_date", "service").distinct().count()
print("Balance de registros y claves Gold: OK")

Balance de registros y claves Gold: OK


## 17. Repetir el streaming

Repito la misma escritura con su checkpoint. El conteo y la cantidad de IDs únicos deben seguir iguales.

In [17]:
antes = spark.read.parquet(BRONZE + "/usage_events").count()
consulta = (flujo.writeStream.format("parquet").partitionBy("event_date")
            .option("path", BRONZE + "/usage_events")
            .option("checkpointLocation", CHECKPOINT + "/eventos")
            .outputMode("append").trigger(availableNow=True).start())
consulta.awaitTermination()
despues = spark.read.parquet(BRONZE + "/usage_events")
assert antes == despues.count() == despues.select("event_id").distinct().count()
print("Antes:", antes, "Después:", despues.count(), "Sin duplicados")

Antes: 43200 Después: 43200 Sin duplicados


## 18. Repetir la escritura de Gold

Uso overwrite para reconstruir el mart, en lugar de agregar otra copia.

In [18]:
antes_gold = spark.read.parquet(GOLD + "/org_daily_usage_by_service").count()
gold.write.mode("overwrite").partitionBy("event_date").parquet(GOLD + "/org_daily_usage_by_service")
despues_gold = spark.read.parquet(GOLD + "/org_daily_usage_by_service").count()
assert antes_gold == despues_gold
print("Gold antes:", antes_gold, "Gold después:", despues_gold)

Gold antes: 11050 Gold después: 11050


## 19. Probar un evento demasiado tardío

En una carpeta temporal envío A, su duplicado, B que avanza el tiempo y un evento antiguo. El watermark es de 10 minutos. Esta prueba no toca el dataset principal.

In [19]:
import tempfile
PRUEBA = tempfile.mkdtemp(prefix="md2_tardios_")
os.makedirs(PRUEBA + "/entrada")
mini = (spark.readStream.schema("event_id STRING, hora TIMESTAMP").json(PRUEBA + "/entrada")
        .withWatermark("hora", "10 minutes").dropDuplicatesWithinWatermark(["event_id"]))
q = (mini.writeStream.format("parquet").option("path", PRUEBA + "/salida")
     .option("checkpointLocation", PRUEBA + "/checkpoint").start())
pruebas = [("A", "10:00"), ("A", "10:01"), ("B", "10:30"), ("TARDIO", "10:00")]
try:
    for i, (identificador, hora) in enumerate(pruebas):
        evento = {"event_id": identificador, "hora": "2025-08-01T" + hora + ":00Z"}
        with open(PRUEBA + f"/entrada/{i}.json", "w") as archivo:
            json.dump(evento, archivo)
        q.processAllAvailable()
finally:
    q.stop()
ids = sorted(fila.event_id for fila in spark.read.parquet(PRUEBA + "/salida").select("event_id").collect())
assert ids == ["A", "B"]
print("IDs que quedaron:", ids)

IDs que quedaron: ['A', 'B']


## 20. Guardar evidencia

Guardo los conteos y muestro una partición y los tamaños. Los descartados demasiado tardíos quedan disponibles en Landing para un futuro reproceso. El dataset original tiene IDs y fechas válidos; esa condición debe revisarse si se cambia la fuente.

In [20]:
evidencia = {"clientes": clientes.count(), "usuarios": usuarios.count(), "facturas": facturas.count(),
             "eventos_bronze": antes, "eventos_silver": silver_guardado.count(),
             "quarantine": quarantine.count(), "gold": antes_gold,
             "repeticion_sin_duplicados": True, "prueba_tardios": ids,
             "astra": "pendiente de ejecutar con mi cuenta"}
with open(EVIDENCIA + "/procesamiento.json", "w") as archivo:
    json.dump(evidencia, archivo, indent=2)
from pathlib import Path
parquets = list(Path(BASE).rglob("*.parquet"))
print("Archivos Parquet:", len(parquets), "MB:", round(sum(p.stat().st_size for p in parquets)/1024**2, 2))
print("Ejemplo de partición:", next(Path(GOLD).rglob("event_date=*")))

Archivos Parquet: 2638 MB: 23.55
Ejemplo de partición: /content/md2_jose/datalake/gold/org_daily_usage_by_service/event_date=2025-08-05


## 21. Conectar con mi base Astra

Necesito un keyspace cloud_analytics creado desde la interfaz de Astra, su Secure Connect Bundle y un token propio. No uso los accesos de mis compañeros.

In [21]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from getpass import getpass
bundle_subido = files.upload()
bundle = os.path.abspath(next(iter(bundle_subido)))
token = getpass("Token de mi base: ")
cluster = Cluster(cloud={"secure_connect_bundle": bundle},
                  auth_provider=PlainTextAuthProvider("token", token))
sesion = cluster.connect("cloud_analytics")
sesion.default_timeout = 120
token = ""
print("Conexión OK")

Saving secure-connect-mineria-datos-jose.zip to secure-connect-mineria-datos-jose.zip
Token de mi base: ··········


Conexión OK


## 22. Crear las tablas

La primera responde por fechas. La segunda guarda un ranking ya calculado. Cassandra no ordena directamente por una suma.

In [22]:
sesion.execute("""CREATE TABLE IF NOT EXISTS cloud_analytics.org_daily_usage_by_service (
 org_id text, event_date date, service text, daily_cost_usd double, requests double,
 genai_tokens bigint, carbon_kg double, event_count bigint,
 PRIMARY KEY ((org_id), event_date, service))""")
sesion.execute("""CREATE TABLE IF NOT EXISTS cloud_analytics.top_services_14d (
 org_id text, fecha_corte date, costo_14d double, service text,
 PRIMARY KEY ((org_id, fecha_corte), costo_14d, service))
 WITH CLUSTERING ORDER BY (costo_14d DESC, service ASC)""")

## 23. Preparar el ranking

Uso los últimos 14 días del dataset, no la fecha actual.

In [23]:
from datetime import timedelta
fecha_corte = gold.agg(F.max("event_date")).first()[0]
inicio = fecha_corte - timedelta(days=13)
ranking = (gold.filter(F.col("event_date").between(inicio, fecha_corte))
           .groupBy("org_id", "service").agg(F.round(F.sum("daily_cost_usd"), 6).alias("costo_14d"))
           .withColumn("fecha_corte", F.lit(fecha_corte)))

## 24. Cargar todo Gold

Uso la función del driver para enviar varias filas a la vez. No genero INSERT para copiar manualmente. collect se usa aquí porque el mart académico es pequeño.

In [24]:
from cassandra.concurrent import execute_concurrent_with_args
insert_diario = sesion.prepare("INSERT INTO cloud_analytics.org_daily_usage_by_service (org_id,event_date,service,daily_cost_usd,requests,genai_tokens,carbon_kg,event_count) VALUES (?,?,?,?,?,?,?,?)")
insert_ranking = sesion.prepare("INSERT INTO cloud_analytics.top_services_14d (org_id,fecha_corte,costo_14d,service) VALUES (?,?,?,?)")
datos_diarios = [tuple(fila) for fila in gold.select("org_id", "event_date", "service", "daily_cost_usd", "requests", "genai_tokens", "carbon_kg", "event_count").collect()]
datos_ranking = [tuple(fila) for fila in ranking.select("org_id", "fecha_corte", "costo_14d", "service").collect()]
execute_concurrent_with_args(sesion, insert_diario, datos_diarios, concurrency=20)
execute_concurrent_with_args(sesion, insert_ranking, datos_ranking, concurrency=20)
print("Filas enviadas:", len(datos_diarios), "Ranking:", len(datos_ranking))

Filas enviadas: 11050 Ranking: 262


## 25. Repetir la carga

Los INSERT con la misma clave actualizan la fila existente. Comparo el conteo antes y después; esta prueba usa la misma entrada.

In [25]:
antes_cql = sesion.execute("SELECT count(*) FROM cloud_analytics.org_daily_usage_by_service").one()[0]
execute_concurrent_with_args(sesion, insert_diario, datos_diarios, concurrency=20)
despues_cql = sesion.execute("SELECT count(*) FROM cloud_analytics.org_daily_usage_by_service").one()[0]
assert antes_cql == despues_cql
print("Cassandra antes:", antes_cql, "Después:", despues_cql)

Cassandra antes: 11050 Después: 11050


## 26. Hacer las dos consultas

Consulto costos y requests en un rango de fechas y los cinco servicios más costosos de 14 días. Guardo resultados reales para la entrega.

In [26]:
org = gold.select("org_id").distinct().orderBy("org_id").first()[0]
cql_1 = "SELECT event_date,service,daily_cost_usd,requests FROM cloud_analytics.org_daily_usage_by_service WHERE org_id=%s AND event_date >= %s AND event_date <= %s"
cql_2 = "SELECT service,costo_14d FROM cloud_analytics.top_services_14d WHERE org_id=%s AND fecha_corte=%s LIMIT 5"
resultado_1 = list(sesion.execute(cql_1, (org, inicio, fecha_corte)))
resultado_2 = list(sesion.execute(cql_2, (org, fecha_corte)))
print("Organización:", org, "Desde:", inicio, "Hasta:", fecha_corte)
for fila in resultado_1: print(fila)
print("Ranking:")
for fila in resultado_2: print(fila)
consultas = {"org_id": org, "desde": str(inicio), "hasta": str(fecha_corte),
            "cql_1": cql_1, "cql_2": cql_2,
            "resultado_1": [fila._asdict() for fila in resultado_1],
            "resultado_2": [fila._asdict() for fila in resultado_2]}
with open(EVIDENCIA + "/consultas_cassandra.json", "w") as archivo:
    json.dump(consultas, archivo, indent=2, default=str)
evidencia["astra"] = "conexión, carga y dos consultas verificadas"
evidencia["cassandra_antes"] = antes_cql
evidencia["cassandra_despues"] = despues_cql
evidencia["ranking_enviado"] = len(datos_ranking)
with open(EVIDENCIA + "/procesamiento.json", "w") as archivo:
    json.dump(evidencia, archivo, indent=2)
cluster.shutdown()

Organización: org_0lvsnujz Desde: 2025-08-18 Hasta: 2025-08-31
Row(event_date=Date(20318), service='database', daily_cost_usd=1.3853, requests=0.0)
Row(event_date=Date(20319), service='analytics', daily_cost_usd=36.5707, requests=475.0)
Row(event_date=Date(20319), service='database', daily_cost_usd=12.3969, requests=245.0)
Row(event_date=Date(20319), service='storage', daily_cost_usd=3.9872, requests=260.0)
Row(event_date=Date(20320), service='database', daily_cost_usd=24.7141, requests=484.0)
Row(event_date=Date(20320), service='storage', daily_cost_usd=9.4223, requests=478.0)
Row(event_date=Date(20321), service='database', daily_cost_usd=12.7963, requests=219.0)
Row(event_date=Date(20321), service='storage', daily_cost_usd=5.9824, requests=350.0)
Row(event_date=Date(20322), service='analytics', daily_cost_usd=48.1036, requests=634.0)
Row(event_date=Date(20322), service='database', daily_cost_usd=1.759, requests=0.0)
Row(event_date=Date(20322), service='storage', daily_cost_usd=2.6725

## 27. Descargar los resultados

Descargo las capas, checkpoints y evidencia después de ejecutar todos los pasos.
El token y el bundle quedan fuera. También descargo el notebook desde Archivo >
Descargar > Descargar .ipynb para conservar sus salidas.

In [27]:
import shutil
carpeta_salida = tempfile.mkdtemp(prefix="resultados_md2_")
for carpeta in ["datalake/bronze", "datalake/silver", "datalake/gold", "datalake/quarantine", "checkpoints", "evidencia"]:
    shutil.copytree(BASE + "/" + carpeta, carpeta_salida + "/" + carpeta, dirs_exist_ok=True)
zip_resultados = shutil.make_archive("/content/Resultados_MD2_Jose", "zip", carpeta_salida)
files.download(zip_resultados)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>